# Can text models predict word-level EEG? (English ZuCo + Persian TeCo)

This notebook is the prerequisite for EEG-shaped embeddings (version A) and brain-tuning (version B). Both can only help if a text model's word vectors predict word EEG **on sentences they never saw**.

For each model and each layer: ridge regression from word vectors to the word's EEG (top 32 principal components, averaged over readers). Folds are split by sentence, and the ridge penalty is also chosen by sentence, so a test sentence never leaks into training. Control: the same pipeline with the EEG shuffled across words.

* **centered:** only differences between words inside a sentence (the cleanest test).
* **raw:** also differences between sentences.
* A layer **passes** when its R² and its margin over shuffled both have 95% CIs above 0.

Models: LaBSE, XLM-R-large, multilingual-E5-large (encoders) and Qwen2.5-1.5B (the decoder used in your LoRA runs; its word vector is the hidden state at the word's last sub-token). All are multilingual, so English and Persian can be compared in the same space later.

**Runtime (GPU):** about 30–45 min for ZuCo, less for TeCo. Progress bars show each model's extraction and its ridge scan; finished models are saved and skipped if Colab disconnects.

## 1. Repository and dependencies

In [ ]:
import os
REPO_URL = 'https://github.com/MohammadJRanjbar/zuco-multimodal-sentiment.git'
BRANCH = 'feature/neurolm-eeg-probe'
REPO_DIR = '/content/zuco-multimodal-sentiment'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin && git -C $REPO_DIR checkout $BRANCH && git -C $REPO_DIR -c user.name=colab -c user.email=colab@users.noreply.github.com pull --rebase origin $BRANCH
%cd $REPO_DIR
!pip install -q -r requirements-diagnostics.txt
!git log --oneline -1

## 2. Drive, paths and settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
THESIS_ROOT = '/content/drive/MyDrive/Thesis'
ARTIFACTS = f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment'
WORD_EEG_DIR = f'{ARTIFACTS}/word_eeg/TRT'                         # ZuCo word EEG (diagnostics notebook)
TECO_TRT_DIR = f'{THESIS_ROOT}/Data/TRT_Total'                      # TeCo <Name>_trt_total.pickle files
TECO_LABELS = f'{THESIS_ROOT}/Data/teco_sentiment_labels_task1.csv'
OUT = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/encoding_scan'
VECTOR_CACHE = '/content/vector_cache'                              # big; kept off Drive
os.environ['HF_HOME'] = f'{ARTIFACTS}/hf_cache'                     # model downloads persist on Drive
os.makedirs(OUT, exist_ok=True)

LABSE_DRIVE = f'{ARTIFACTS}/models/LaBSE'
MODELS = ['labse' + (f'={LABSE_DRIVE}' if os.path.exists(f'{LABSE_DRIVE}/config.json') else ''),
          'xlmr-large', 'me5-large', 'qwen2.5-1.5b']
LAYER_STRIDE = 1        # 2 = every other layer (about half the time)
MODEL_ARGS = ' '.join(MODELS)
DEVICE = 'cuda' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu'
print('device:', DEVICE, '| models:', MODEL_ARGS)
print('ZuCo cache found:', os.path.exists(WORD_EEG_DIR), '| TeCo pickles found:', os.path.exists(TECO_TRT_DIR),
      '| TeCo labels found:', os.path.exists(TECO_LABELS))

## 3. Tests (about 10 s)

In [ ]:
!python -m pytest -q tests/test_encoding_scan.py

## 4. Step 1 — English (ZuCo)

Each model shows two progress bars: word-vector extraction (`batch`), then the ridge scan (`fold` = one layer × mode × fold), with the best R² so far.

In [ ]:
!python scripts/scan_eeg_encoding.py --dataset zuco --word-eeg-dir "$WORD_EEG_DIR" --out-dir "$OUT" \
    --vector-cache-dir "$VECTOR_CACHE" --models $MODEL_ARGS --layer-stride $LAYER_STRIDE --device $DEVICE

In [ ]:
from IPython.display import Image, Markdown, display
def show(dataset):
    folder = f'{OUT}/encoding_scan_{dataset}'
    display(Markdown(open(f'{folder}/encoding_scan_{dataset}.md').read()))
    display(Image(f'{folder}/encoding_scan_{dataset}.png'))
show('zuco')

## 5. Step 2 — Persian (TeCo)

First check that the pickles look as expected: word text, and a `TRT_total` vector of 126 values per fixated word. If loading fails, the error says which keys it found; paste it to me.

In [ ]:
import glob
from src.brainshaping.teco import describe_pickle, load_teco_trials
from src.brainshaping.data import item_eeg, sentence_table
print(describe_pickle(sorted(glob.glob(f'{TECO_TRT_DIR}/*_trt_total.pickle'))[0]))
trials = load_teco_trials(TECO_TRT_DIR, TECO_LABELS)
items, eeg = item_eeg(trials, drop_channels=())
sentences = sentence_table(trials)
print(f"{len({t['subject_id'] for t in trials})} readers, {len(sentences)} sentences, {len(items)} words with EEG, "
      f"{eeg.shape[1]} EEG values per word")
print('labels (0 neg, 1 neu, 2 pos):', sentences['label_id'].value_counts().sort_index().to_dict())
print('example:', ' '.join(sentences['words'].iloc[0]))
del trials, items, eeg

In [ ]:
!python scripts/scan_eeg_encoding.py --dataset teco --trt-dir "$TECO_TRT_DIR" --labels-csv "$TECO_LABELS" \
    --out-dir "$OUT" --vector-cache-dir "$VECTOR_CACHE" --models $MODEL_ARGS --layer-stride $LAYER_STRIDE --device $DEVICE

In [ ]:
show('teco')

## 6. How to read the result

* **PASS for a language:** some layers predict that language's word EEG on new sentences.
* **FAIL:** no layer beats the average; amplifying "EEG directions" would amplify noise for that language.
* Look for **several neighbouring layers** passing, not one; and whether the same models/depths pass in both languages.

**Controls (bottom of each report):** for each model's best layer, the scan checks whether the text vectors predict anything about EEG that simple word features don't:
* *lexical*: word length, frequency (wordfreq, English or Persian), position in the sentence, punctuation;
* *lexical + reading*: also how many readers fixated the word, number of fixations, reading time (ZuCo only).
* **Beyond lexical: signal remains** → the text models carry brain-relevant information beyond length/frequency, which is what EEG-shaped embeddings would need. **Nothing remains** → the "EEG directions" are length/frequency directions, which cannot add sentiment information.
* **Noise ceiling:** the share of word-to-word EEG variance that repeats across readers; no model can exceed it.

Re-running the scan cells reuses the finished model scans and only computes the controls (a few minutes).

## Save results to GitHub

Reports, CSVs and plots go to `saved_results/encoding_scan/` on your fork (the word-vector cache stays in `/content` and is not saved). Needs the `GITHUB_TOKEN` Colab secret (fine-grained token, *Contents: Read and write* on your fork).

In [ ]:
from google.colab import userdata
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
GIT_NAME, GIT_EMAIL = 'mohammadranjbar', 'mjrkalhar@gmail.com'
!python scripts/save_results_to_github.py --name encoding_scan --source "$OUT" --push \
    --author-name "$GIT_NAME" --author-email "$GIT_EMAIL"